# Module 7: UDFs and Complex Types

Spark has 300+ built-in functions, but sometimes you need **custom logic**. That's where User Defined Functions (UDFs) come in.

You'll also learn to work with **complex column types** — arrays, structs, and maps — which represent nested and semi-structured data.

**Performance warning**: PySpark UDFs serialize data between the JVM and Python for every row. This is **slow**. Always prefer built-in functions. When you must use custom logic, use Pandas UDFs (vectorized) over regular UDFs.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, udf, array, explode, collect_list, size, struct, create_map, map_keys, map_values
from pyspark.sql.types import StringType, FloatType

spark = SparkSession.builder \
    .appName("Module 07 - UDFs and Complex Types") \
    .master("local[*]") \
    .getOrCreate()

employees = spark.read.csv("../data/employees.csv", header=True, inferSchema=True)
departments = spark.read.csv("../data/departments.csv", header=True, inferSchema=True)
sales = spark.read.csv("../data/sales.csv", header=True, inferSchema=True)

---
## Concept 1: Simple UDFs with @udf Decorator

A UDF wraps a Python function so Spark can apply it to every row. You must specify the return type.

In [ ]:
@udf(returnType=StringType())
def salary_grade(salary):
    """Assign a letter grade based on salary."""
    if salary is None:
        return None
    if salary >= 100000:
        return "A"
    elif salary >= 80000:
        return "B"
    else:
        return "C"

employees.withColumn("grade", salary_grade(col("salary"))) \
    .select("name", "salary", "grade") \
    .show(10)

#### Try It: Write a Sale Size UDF

Create a UDF `sale_size` that categorizes sale amounts:
- `"Small"` if amount < 500
- `"Medium"` if 500 <= amount <= 1500
- `"Large"` if amount > 1500

Apply it to the sales DataFrame and count how many sales fall into each category.

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
from pyspark.sql.functions import count

@udf(returnType=StringType())
def sale_size(amount):
    if amount is None:
        return None
    if amount < 500:
        return "Small"
    elif amount <= 1500:
        return "Medium"
    else:
        return "Large"

sales.withColumn("size", sale_size(col("amount"))) \
    .groupBy("size").agg(count("*").alias("count")) \
    .orderBy("count", ascending=False).show()

---
## Concept 2: Registering UDFs for SQL

To use a UDF inside `spark.sql()`, register it with `spark.udf.register()`.

In [ ]:
spark.udf.register(
    "salary_grade_sql",
    lambda s: "A" if s and s >= 100000 else ("B" if s and s >= 80000 else "C"),
    StringType()
)

employees.createOrReplaceTempView("employees")

spark.sql("""
    SELECT name, salary, salary_grade_sql(salary) AS grade
    FROM employees
    ORDER BY salary DESC
    LIMIT 10
""").show()

#### Try It: Register Your Sale Size UDF for SQL

1. Register a UDF called `sale_size_sql` that does the same Small/Medium/Large categorization
2. Register sales as a temp view
3. Write a SQL query that uses it: `SELECT product, amount, sale_size_sql(amount) AS size FROM sales LIMIT 10`

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
spark.udf.register(
    "sale_size_sql",
    lambda a: "Small" if a and a < 500 else ("Medium" if a and a <= 1500 else "Large"),
    StringType()
)

sales.createOrReplaceTempView("sales")

spark.sql("""
    SELECT product, amount, sale_size_sql(amount) AS size
    FROM sales
    ORDER BY amount DESC
    LIMIT 10
""").show()

---
## Concept 3: Arrays — collect_list, size, explode

An **array column** holds a list of values. Key operations:
- `collect_list(col)` — aggregate values into an array (during groupBy)
- `size(col)` — length of the array
- `explode(col)` — flatten array back into individual rows

**Real-world use**: Collecting all products a customer bought, all tags on a post, all events in a session.

In [ ]:
# Group employees by city, collect names into an array
city_teams = employees.groupBy("city").agg(
    collect_list("name").alias("team_members")
)
city_teams.show(truncate=False)

In [ ]:
# Array size
city_teams.withColumn("team_size", size("team_members")).show(truncate=False)

In [ ]:
# Explode: flatten array back to individual rows
city_teams.select("city", explode("team_members").alias("member")).show(10)

#### Try It: Products Per Employee

1. Join sales to employees (on `employee_id`)
2. Group by `name`
3. Use `collect_list("product")` to get all products each employee sold
4. Add a `num_sales` column using `size()`
5. Show the top 5 by `num_sales`

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
from pyspark.sql.functions import desc

employee_products = (
    sales.join(employees.select("employee_id", "name"), on="employee_id")
    .groupBy("name")
    .agg(collect_list("product").alias("products_sold"))
    .withColumn("num_sales", size("products_sold"))
    .orderBy(desc("num_sales"))
)
employee_products.show(5, truncate=False)

---
## Concept 4: Structs and Maps

**Struct**: A nested object with named fields — like a row within a column. Access fields with dot notation.

**Map**: A key-value dictionary column. Access values with bracket notation.

In [ ]:
# Struct: nest multiple columns into one
nested = employees.select(
    "employee_id",
    struct("name", "city", "salary").alias("details")
)
nested.show(5, truncate=False)
nested.printSchema()

In [ ]:
# Access struct fields with dot notation
nested.select("employee_id", "details.name", "details.salary").show(5)

In [ ]:
# Map: create a key-value dictionary column
emp_map = employees.select(
    "employee_id",
    create_map(lit("name"), col("name"), lit("city"), col("city")).alias("info")
)
emp_map.show(5, truncate=False)

# Access map values with bracket notation
emp_map.select("employee_id", col("info")["name"].alias("name")).show(5)

#### Try It: Create a Struct

1. Join employees to departments
2. Create a struct column called `profile` containing: `name`, `department_name`, `salary`
3. Select `employee_id` and `profile`
4. Access `profile.name` and `profile.salary` using dot notation

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
emp_dept = employees.join(departments, on="department_id")

profiled = emp_dept.select(
    "employee_id",
    struct("name", "department_name", "salary").alias("profile")
)
profiled.show(5, truncate=False)

profiled.select("employee_id", "profile.name", "profile.salary").show(5)

---
## Concept 5: Pandas UDFs (Vectorized)

Pandas UDFs operate on **batches** of data using pandas Series, which is **10-100x faster** than regular UDFs. They use Apache Arrow for zero-copy data transfer between the JVM and Python.

**Rule**: If you must write a custom function, always prefer Pandas UDFs over regular UDFs for numeric operations.

In [ ]:
from pyspark.sql.functions import pandas_udf
import pandas as pd

@pandas_udf("double")
def salary_tax(salary: pd.Series) -> pd.Series:
    """Calculate 30% tax on salary (vectorized — much faster)."""
    return salary * 0.30

employees.withColumn("tax", salary_tax(col("salary"))) \
    .select("name", "salary", "tax") \
    .show(10)

#### Try It: Write a Pandas UDF

Write a Pandas UDF `annual_bonus` that computes 15% of salary. Apply it to the employees DataFrame.

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
@pandas_udf("double")
def annual_bonus(salary: pd.Series) -> pd.Series:
    return salary * 0.15

employees.withColumn("bonus", annual_bonus(col("salary"))) \
    .select("name", "salary", "bonus") \
    .show(10)

---
## Performance Reference

| Approach | Speed | Use When |
|----------|-------|----------|
| **Built-in functions** | Fastest | Always prefer (`when`, `regexp_replace`, math ops) |
| **Pandas UDF** | Fast | Custom numeric/batch logic |
| **Regular UDF** | Slowest | Simple custom logic, last resort |

---
## Capstone Exercise

Combine UDFs, arrays, and structs:

1. Apply the `sale_size` UDF to categorize each sale
2. Join sales (with size) to employees
3. Group by employee name, collect all sale sizes into an array
4. Add a `total_sales` column (size of the array)
5. Show results ordered by total_sales descending

In [ ]:
# Your capstone code here


In [ ]:
# --- Capstone Solution ---
sales_categorized = sales.withColumn("sale_size", sale_size(col("amount")))

result = (
    sales_categorized
    .join(employees.select("employee_id", "name"), on="employee_id")
    .groupBy("name")
    .agg(collect_list("sale_size").alias("sale_categories"))
    .withColumn("total_sales", size("sale_categories"))
    .orderBy(desc("total_sales"))
)

result.show(10, truncate=False)

In [ ]:
spark.stop()

---
## What You Learned

- **`@udf`** wraps Python functions for row-by-row application (slow but flexible)
- **`spark.udf.register()`** makes UDFs available in SQL
- **Arrays**: `collect_list` aggregates, `size` measures, `explode` flattens
- **Structs**: nested objects with dot-notation access
- **Maps**: key-value dictionaries with bracket-notation access
- **Pandas UDFs** (`@pandas_udf`) are 10-100x faster than regular UDFs
- **Always prefer built-in functions** over any UDF

Next: **Module 8 — Spark with Scala**, where you'll see all of this in Spark's native language.